# 00 - Start here

**What this project is.** A merchant's fraud stack blocks an order. Nobody ever
looks at that decision again, and no row is written anywhere saying "we just
refused a good customer." This system opens a case on every blocked order,
pulls cross-merchant evidence the merchant cannot see, and decides whether the
block should stand.

**What is built right now.** Phase 1 -- the deterministic decision core -- is
complete. There is no LLM anywhere in this codebase yet. Phase 2, the agent
that writes verdicts and runs verification dialogues, is not started.

| Layer | Module | Job |
|---|---|---|
| Data | `datagen/generate.py` | Invents a year of payment traffic and a risk stack that blocks some of it |
| Evidence | `core/feature_store.py` | Assembles one case's evidence in three named blocks |
| Answer key | `core/truth.py` | The only door to ground truth, and it is locked during training |
| Model | `core/model.py` | P(this block was correct), calibrated |
| Policy | `core/policy.py` | Turns that probability into a decision using money |
| Replay | `core/backtest.py` | Runs the year chronologically and writes a ledger |
| Grading | `core/metrics.py` | Joins truth afterwards and prices every decision |
| Report | `core/report.py` | Writes `METRICS.md` |

The rest of these notebooks walk the chain in order. Nothing below is typed by
hand -- every number is computed live when the notebook runs.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.environ['PYTHONUTF8'] = '1'
import numpy as np, warnings
warnings.filterwarnings('ignore')
DATA = '../data300k'      # the working set
DATA100K = '../data'      # the calibrated baseline the pitch quotes

In [2]:
from core.feature_store import FeatureStore
from core.truth import TruthVault
from core.model import Adjudicator
from core.policy import PolicyConfig
from core.backtest import run
from core.metrics import grade, StepUpModel

store = FeatureStore.load(DATA)
vault = TruthVault(DATA)
print(store)
print(f'answer key holds {len(vault):,} payments (only ~{len(store):,} of them were blocked)')

<FeatureStore 8265 cases train=6490 holdout=1775>
answer key holds 300,000 payments (only ~8,265 of them were blocked)


## The whole pipeline, in one cell

Load evidence, score it, decide, replay, grade. Five lines. Everything after
this notebook is an explanation of one of them.

In [3]:
model  = Adjudicator().fit(store, vault)          # 1. learn, on train only
cfg    = PolicyConfig(cap=0.02)                    # 2. an operating point
ledger = run(store, model, cfg, 'holdout')         # 3. replay the blocked pile
out    = grade(ledger, vault, StepUpModel())       # 4. join truth, price it

print(f'{out.n_cases} appealed cases -> {out.n_released} released')
print(f'precision {out.precision:.1%}   recall of recoverable {out.recall_recoverable:.1%}')
print(f'recovered Rs {out.recovered_inr/1e7:.2f} cr')
print(f'fraud admitted Rs {out.fraud_admitted_inr/1e5:.2f} L')
print(f'net contribution Rs {out.net_contribution_inr/1e7:.2f} cr')
print(f'abstained on {out.abstention_rate:.1%} of cases')

1775 appealed cases -> 1181 released
precision 98.6%   recall of recoverable 81.1%
recovered Rs 7.04 cr
fraud admitted Rs 22.57 L
net contribution Rs 1.53 cr
abstained on 25.9% of cases


That last line matters as much as the first. The system refuses to decide on a
meaningful share of cases, and that is a designed behaviour, not a gap. See
notebook 03.